# Confronto tra descrittori strutturali e feature MD

- **Dati**: campioni con una proprietà target e, quando disponibili, feature ottenute dalle simulazioni MD.
- **Modello**: Gradient Boosting.
- **Valutazione**: nested cross-validation per le prestazioni e per le spiegazioni SHAP out-of-fold. Il singolo split serve solo per controlli e grafici esplorativi.

Le impostazioni principali sono nella prima cella: MODE decide quali dati usare, USE_MORDRED aggiunge descrittori 2D e MD_FEATURES sceglie quali dati MD caricare. Per i dati a più temperature, le modalità 250_delta e 300_delta combinano un valore di riferimento con la variazione osservata tra le temperature.


In [ ]:
import hashlib
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

warnings.filterwarnings("ignore")

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()
while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent
if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
MD_DATA_DIR = ROOT / "data" / "md_features"
RUNS_DIR = ROOT / "runs"


# Modalita di esecuzione: quali dati usare per prevedere il target
#   "struct"        -> solo descrittori strutturali (niente MD)
#   "md_only"      -> solo feature MD
#   "struct+md"     -> descrittori strutturali e feature MD
MODE = "struct+md"

# Feature MD da caricare (ignorate con MODE == "struct")
#   "single"                  -> dati da una simulazione per campione (data/md_features/single_chain.csv)
#   "300" / "250" / "350"     -> una sola temperatura
#   "multitemp_300_delta"     -> valore a 300 K + variazione (350-250)/2
#   "multitemp_250_delta"     -> valore a 250 K + variazione non scalata 350-250 [DEFAULT]
#   "multitemp_temperatures"  -> tutti i valori a 250, 300, 350 K
MD_FEATURES = "multitemp_300_delta"

# Descrittori 2D aggiuntivi di Mordred
# (ignorato se MODE == "md_only", perchÃ¨ in quel caso non usiamo descrittori strutturali)
USE_MORDRED = False

# Etichetta opzionale da riportare nel log
RUN_TAG = ""

# Sorgente dati e seed
target = "Tg"
mongo_db = "PolymerPrediction"
mongo_collection = "biceranoPolymers"
random_state = 28

# Soglie usate per filtrare e selezionare le feature
MIN_PREV = 0.02   # frazione min/max di accensione di un bit per tenerlo
MAX_NAN = 0.20    # oltre questa frazione di NaN sul train una colonna si butta
N_MI = 50         # quante feature sopravvivono allo stadio 1 della selezione

# Parametri per UMAP e clustering
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_K_VALUES = [2, 3, 4, 5]

# ============================================================================
# Impostazioni derivate dai parametri scelti sopra
# ============================================================================

# Controlla la modalita selezionata
if MODE not in {"struct", "md_only", "struct+md"}:
    raise ValueError(f"MODE sconosciuto: {MODE!r}. Usa 'struct', 'md_only' o 'struct+md'")

# Controlla le feature MD richieste
valid_md = {"single", "300", "250", "350", "multitemp_300_delta", "multitemp_250_delta", "multitemp_temperatures"}
if MD_FEATURES not in valid_md:
    raise ValueError(f"MD_FEATURES sconosciuto: {MD_FEATURES!r}. Usa uno di: {valid_md}")

# Ricava i flag usati dal resto del notebook
USE_MD = MODE != "struct"
MD_ONLY = MODE == "md_only"

# In modalità MD-only non si usano descrittori strutturali
if MD_ONLY:
    USE_MORDRED = False

# Traduce MD_FEATURES nelle impostazioni di caricamento
if MD_FEATURES == "single":
    MD_DATASET = "single"
    MD_MULTITEMP_MODE = None
    md_source = str(MD_DATA_DIR / "single_chain.csv")
elif MD_FEATURES in {"300", "250", "350"}:
    MD_DATASET = MD_FEATURES
    MD_MULTITEMP_MODE = None
    md_source = os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_{MD_FEATURES}.csv")
elif MD_FEATURES.startswith("multitemp_"):
    MD_DATASET = "multitemp"
    MD_MULTITEMP_MODE = MD_FEATURES[10:]  # "300_delta", "250_delta", "temperatures"
    md_source = " + ".join([
        os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_base.csv"),
        os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_250.csv"),
        os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_350.csv"),
    ])
else:
    raise ValueError(f"MD_FEATURES non supportato: {MD_FEATURES}")

# Definisce i modelli da confrontare
if MD_ONLY:
    ARMS = ["md_only"]
    ARM_LABEL = {"md_only": "MD-only"}
elif not USE_MD:
    ARMS = ["base"]
    ARM_LABEL = {"base": "base (solo strutturali)"}
else:
    ARMS = ["base", "full"]
    ARM_LABEL = {"base": "base (solo strutturali)", "full": "full (strutturali + MD)"}

# Prepara i file e le colonne MD
if not USE_MD:
    md_feats = []
else:
    # File e colonne disponibili per ciascuna temperatura
    MD_TEMPS = {
        "300": (os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_base.csv"), ["Rg_md", "density_md", "end_to_end_md"]),
        "250": (os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_250.csv"),  ["Rg_md_250", "density_md_250", "end_to_end_md_250"]),
        "350": (os.path.join(MD_DATA_DIR, f"{mongo_collection}_analysis_summary_350.csv"),  ["Rg_md_350", "density_md_350", "end_to_end_md_350"]),
    }
    MD_MULTITEMP_PROPERTIES = ["Rg_md", "density_md", "end_to_end_md"]
    
    if MD_DATASET == "single":
        md_files = [MD_DATA_DIR / "single_chain.csv"]
        md_feats = ["Rg", "tau", "trans", "gauche", "cis", "mean_angle", "var_dihedral", "circular_var"]
    elif MD_DATASET == "multitemp":
        if MD_MULTITEMP_MODE == "300_delta":
            md_feats = [
                *[f"{prop}_300" for prop in MD_MULTITEMP_PROPERTIES],
                *[f"{prop}_delta" for prop in MD_MULTITEMP_PROPERTIES],
            ]
        elif MD_MULTITEMP_MODE == "250_delta":
            md_feats = [
                *[f"{prop}_250" for prop in MD_MULTITEMP_PROPERTIES],
                *[f"{prop}_delta" for prop in MD_MULTITEMP_PROPERTIES],
            ]
        else:  # "temperatures"
            md_feats = [
                *[f"{prop}_250" for prop in MD_MULTITEMP_PROPERTIES],
                *[f"{prop}_300" for prop in MD_MULTITEMP_PROPERTIES],
                *[f"{prop}_350" for prop in MD_MULTITEMP_PROPERTIES],
            ]
        md_files = [MD_TEMPS[t][0] for t in ("300", "250", "350")]
    elif MD_DATASET in MD_TEMPS:
        md_files = [MD_TEMPS[MD_DATASET][0]]
        md_feats = list(MD_TEMPS[MD_DATASET][1])
    else:
        raise ValueError(f"MD_DATASET sconosciuto: {MD_DATASET!r}")


def with_md(X_):
    """Attacca le colonne MD a una matrice di feature. No-op se USE_MD = False."""
    return X_.join(df_md[md_feats]) if USE_MD else X_

In [ ]:
# Logging del run. Ogni esecuzione crea runs/<timestamp>__<collection>__<md|nomd>/ con
# config, metriche, tabelle e figure, e aggiunge una riga a runs/index.csv per confrontare i run
# tra loro. Il codice sta in runlog.py.
from runlog import RunLogger, load_runs

log = RunLogger(
    config={
        "db": mongo_db,
        "collection": mongo_collection,
        "target": target,
        "mode": MODE,
        "use_md": USE_MD,
        "md_only": MD_ONLY,
        "md_features": MD_FEATURES if USE_MD else None,
        "md_feats": md_feats,
        "md_source": md_source if USE_MD else None,
        "use_mordred": USE_MORDRED,
        "arms": ARMS,
        "random_state": random_state,
        "min_prevalence": MIN_PREV,
        "n_mi": N_MI,
    },
    tag=RUN_TAG,
)


## 1. Dati

Il notebook legge i campioni dal database e, se richiesto, aggiunge le feature MD dai file presenti in data/md_features. Prima dell'analisi vengono rimossi i campioni senza SMILES, senza valore target o senza le feature MD necessarie.


In [ ]:
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
client = MongoClient(os.getenv("MONGO_URI"))
collection = client[mongo_db][mongo_collection]

df = pd.DataFrame(list(collection.find({})))
n_docs = len(df)   # quanti documenti ci sono nella collection, prima di qualunque filtro
df = pd.concat([df.drop(columns=["properties"]), pd.json_normalize(df["properties"])], axis=1)
df = df.set_index("_id")
df.index = df.index.astype(str)
df = df.sort_index()   # ordine deterministico: rende split e selezione riproducibili tra run MD diversi

if USE_MD:
    # Feature MD: una riga per campione, calcolata a partire dalle simulazioni.
    # Per multitemp si puo' conservare ogni temperatura oppure convertire
    # in un valore unico di riferimento a 300 K o a 250 K e in una variazione termica.
    md_frames = {
        temperature: pd.read_csv(MD_TEMPS[temperature][0], sep=",").set_index("polymer_id")
        for temperature in ("300", "250", "350")
    } if MD_DATASET == "multitemp" else None

    if MD_DATASET == "multitemp":
        md_index = md_frames["300"].index.union(md_frames["250"].index).union(md_frames["350"].index)
        if MD_MULTITEMP_MODE == "300_delta":
            df_md = pd.DataFrame(index=md_index)
            for prop in MD_MULTITEMP_PROPERTIES:
                df_md[f"{prop}_300"] = md_frames["300"][prop]
                df_md[f"{prop}_delta"] = (
                    md_frames["350"][f"{prop}_350"] - md_frames["250"][f"{prop}_250"]
                ) / 2
        elif MD_MULTITEMP_MODE == "250_delta":
            df_md = pd.DataFrame(index=md_index)
            for prop in MD_MULTITEMP_PROPERTIES:
                df_md[f"{prop}_250"] = md_frames["250"][f"{prop}_250"]
                df_md[f"{prop}_delta"] = (
                    md_frames["350"][f"{prop}_350"] - md_frames["250"][f"{prop}_250"]
                )
        else:
            df_md = pd.concat([
                md_frames["250"][MD_TEMPS["250"][1]].rename(
                    columns={source: f"{prop}_250" for source, prop in zip(
                        MD_TEMPS["250"][1], MD_MULTITEMP_PROPERTIES
                    )}
                ),
                md_frames["300"][MD_TEMPS["300"][1]].rename(
                    columns={source: f"{prop}_300" for source, prop in zip(
                        MD_TEMPS["300"][1], MD_MULTITEMP_PROPERTIES
                    )}
                ),
                md_frames["350"][MD_TEMPS["350"][1]].rename(
                    columns={source: f"{prop}_350" for source, prop in zip(
                        MD_TEMPS["350"][1], MD_MULTITEMP_PROPERTIES
                    )}
                ),
            ], axis=1)
    else:
        frames = [pd.read_csv(f, sep=",").set_index("polymer_id") for f in md_files]
        df_md = pd.concat(frames, axis=1)

    df_md.index = df_md.index.astype(str)
    df = df.join(df_md[md_feats], rsuffix="_md")
else:
    df_md = None   # nessun join: le colonne MD non esistono e nessuna cella le usa


def with_md(X_):
    """Attacca le colonne MD a una matrice di feature. No-op se USE_MD = False."""
    return X_.join(df_md[md_feats]) if USE_MD else X_


# Tiene solo i campioni utilizzabili per questa configurazione
df = df.dropna(subset=["smiles", target, *md_feats])
df = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()
dataset_index_signature = hashlib.sha256("\n".join(df.index).encode()).hexdigest()

print(f"{len(df)} polimeri | {target} {df[target].min():.0f}-{df[target].max():.0f} K"
      f" | feature MD: {'si' if USE_MD else 'no'}")

log.dataset(
    db=mongo_db,
    collection=mongo_collection,
    n_docs=n_docs,                 # documenti nella collection
    n_polymers=len(df),            # righe sopravvissute ai filtri (SMILES/Tg/MD presenti e validi)
    n_dropped=n_docs - len(df),
    target=target,
    target_min=float(df[target].min()),
    target_max=float(df[target].max()),
    target_mean=float(df[target].mean()),
    target_std=float(df[target].std()),
    use_md=USE_MD,
    md_source=md_source if USE_MD else None,
    md_feats=md_feats,
    index_signature=dataset_index_signature,
)
log.table("dataset_describe", df[[target, *md_feats]].describe().round(2))

df[[target, *md_feats]].describe().round(2)

## 2. Analisi esplorativa

In [ ]:
if not USE_MD:
    print("USE_MD = False: non ci sono feature MD da correlare, cella saltata")
else:
    cols = [*md_feats, target]
    corr = df[cols].corr(method="spearman")

    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask)] = True

    plt.figure(figsize=(len(cols) * 0.9, len(cols) * 0.75))
    sns.heatmap(
        corr, annot=True, fmt=".2f", vmin=-1, vmax=1,
        cmap="coolwarm", square=True, mask=mask,
        linewidths=0.5, annot_kws={"size": 8}, cbar_kws={"shrink": 0.6},
    )
    plt.title(f"Correlazione Spearman (feature MD vs {target})", fontsize=11, pad=10)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    log.figure("md_correlation")            
    plt.show()

    log.table("md_correlation_spearman", corr.round(3))
    log.result("md_correlation", {
        f"spearman_{f}_vs_{target}": float(corr.loc[f, target]) for f in md_feats
    })

## 3. Descrittori strutturali

Dallo SMILES vengono calcolati descrittori continui RDKit, fingerprint Morgan e chiavi MACCS. Se USE_MORDRED = True, si aggiungono anche i descrittori 2D di Mordred. Queste colonne descrivono la struttura e passano insieme al filtro e alla selezione delle feature. Le feature MD vengono invece tenute separate, perchè il loro contributo viene valutato esplicitamente nel confronto tra i modelli.


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator
from rdkit.ML.Descriptors import MoleculeDescriptors

if not MD_ONLY:
    descriptor_names = [d[0] for d in Descriptors._descList]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

    feature_names = (
        descriptor_names
        + [f"fp_{i}" for i in range(1024)]
        + [f"maccs_{i}" for i in range(167)]   # MACCS ha 167 bit, lo 0 non e usato
    )


    def featurize(smiles):
        """Descrittori continui + Morgan + MACCS per un singolo SMILES."""
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:  # non dovrebbe capitare, gli SMILES invalidi sono gia' scartati
            return pd.Series(np.nan, index=feature_names)
        values = (
            list(calc.CalcDescriptors(mol))
            + list(morgan_gen.GetFingerprint(mol))
            + list(MACCSkeys.GenMACCSKeys(mol))
        )
        return pd.Series(values, index=feature_names)


    feature_df = pd.DataFrame([featurize(smiles) for smiles in df["smiles"]], index=df.index)

    # Gli SMILES che RDKit non parsa diventano righe tutte-NaN. 
    dead_rows = feature_df.index[feature_df.isna().all(axis=1)]
    if len(dead_rows):
        print(f"{len(dead_rows)} SMILES non parsati da RDKit -> righe scartate")
        print("  es:", df.loc[dead_rows[0], "smiles"])
        df = df.drop(index=dead_rows)
        feature_df = feature_df.drop(index=dead_rows)

    log.dataset(n_smiles_non_parsati=len(dead_rows), n_polymers=len(df))

    print("Matrice delle feature:", feature_df.shape)
    feature_df.iloc[:5, :6]
else:
    print("Cella saltata, MD_ONLY=True")

### 3.1 Mordred (opzionale)

Mordred calcola ~1600 descrittori 2D. Molti di questi hanno lo **stesso nome** di un descrittore RDKit (le famiglie `*_VSA`, `EState_*`, `BalabanJ`, `LabuteASA`) e altri hanno gli stessi **valori** sotto un nome diverso (`FCSP3` = `FractionCSP3`, `nHetero` = `NumHeteroatoms`, `Xp-0d` = `Chi0`).

Due colonne identiche si spartiscono l'importanza, quindi ognuna delle due sembra meno importante di quanto sia. Per questo, la cella qui sotto li cerca in due modi e in entrambi i casi tiene la versione RDKit:

1. **stesso nome**: confronta i valori e li registra. Dove coincidono, la copia Mordred si butta.
   Dove **non** coincidono vengono tenuti entrambi.
2. **stessi valori, nome diverso**: si butta la colonna Mordred. Attenzione: "stessi valori"
   e vero *su questo dataset*, non sempre in generale. Non è un problema per
   il modello (colonne identiche qui sono ridondanti qui), ma vuol dire che la lista dei
   descrittori Mordred tenuti può cambiare da dataset a dataset.

Quello che sopravvive viene rinominato `mrd_*`, così nelle tabelle e nei plot SHAP si vede a colpo
d'occhio da dove viene una feature. Il confronto non guarda `y` nè la divisione train/test: è una
verifica di identità tra colonne, non un filtro fittato sui dati, quindi non è leakage.

In [ ]:
if not USE_MORDRED:
    print("USE_MORDRED = False: Mordred non usato, cella saltata")
else:
    from mordred import Calculator, descriptors
    mordred_calc = Calculator(descriptors, ignore_3D=True)

    mols = [Chem.MolFromSmiles(str(s)) for s in df["smiles"]]
    assert all(m is not None for m in mols), "SMILES non parsabile: doveva gia' essere fuori (dead_rows)"

    # nproc=1: la modalita' parallela di mordred usa multiprocessing e su Windows dentro Jupyter si blocca
    mordred_df = mordred_calc.pandas(mols, nproc=1, quiet=True)
    mordred_df.index = df.index

    # I descrittori che mordred non riesce a calcolare non diventano NaN: restano oggetti Error/Missing
    # e tengono la colonna a dtype object, che fa saltare tutto a valle. to_numeric li rende NaN.
    mordred_df = mordred_df.apply(pd.to_numeric, errors="coerce")
    n_mordred_raw = mordred_df.shape[1]

    # colonne non calcolabili su NESSUNA molecola
    mordred_df = mordred_df.dropna(axis=1, how="all")
    n_all_nan = n_mordred_raw - mordred_df.shape[1]

    # rende la cella ri-eseguibile: senza questo, un secondo giro accoda le mrd_* a quelle gia' inserite
    feature_df = feature_df[[c for c in feature_df.columns if not c.startswith("mrd_")]]

    def _sig(col):
        """Firma dei valori di una colonna: due colonne identiche hanno la stessa firma."""
        v = pd.to_numeric(col, errors="coerce").to_numpy(dtype=float)
        return np.round(v, 6).tobytes()

    # 1. duplicati per NOME -----------------------------------------------------------------
    same_name = [c for c in mordred_df.columns if c in feature_df.columns]

    dup_rows = []
    for c in same_name:
        rd, mrd = feature_df[c], mordred_df[c]
        both = rd.notna() & mrd.notna()          # dove NON sono confrontabili non concludo niente
        diff = (rd[both] - mrd[both]).abs()
        dup_rows.append({
            "descrittore": c,
            "n_confrontabili": int(both.sum()),
            "valori_uguali": bool(both.any()) and np.allclose(rd[both], mrd[both], rtol=1e-5, atol=1e-8),
            "max_diff_assoluta": float(diff.max()) if both.any() else np.nan,
        })
    dup_name_df = pd.DataFrame(
        dup_rows, columns=["descrittore", "n_confrontabili", "valori_uguali", "max_diff_assoluta"]
    ).sort_values(["valori_uguali", "descrittore"])   # i conflitti in cima: sono quelli da guardare

    identici = dup_name_df.loc[dup_name_df["valori_uguali"], "descrittore"].tolist()
    conflitti = dup_name_df.loc[~dup_name_df["valori_uguali"], "descrittore"].tolist()
    mordred_df = mordred_df.drop(columns=identici)   # copia esatta: resta quella RDKit

    # 2. duplicati per VALORE, sotto un altro nome ------------------------------------------
    # Le colonne costanti restano fuori dal confronto: sono tutte uguali tra loro (colonne di zeri) e
    # matcherebbero tutte. Le butta comunque il FeatureFilter.
    rdkit_by_sig = {}
    for c in feature_df.columns:
        if feature_df[c].std(ddof=0) > 0:
            rdkit_by_sig.setdefault(_sig(feature_df[c]), []).append(c)

    dup_value_df = pd.DataFrame([
        {"mordred": c, "struct": " ".join(rdkit_by_sig[_sig(mordred_df[c])])}
        for c in mordred_df.columns
        if mordred_df[c].std(ddof=0) > 0 and _sig(mordred_df[c]) in rdkit_by_sig
    ], columns=["mordred", "struct"])
    mordred_df = mordred_df.drop(columns=dup_value_df["mordred"].tolist())

    # 3. quello che resta entra nel pool, con la provenienza scritta nel nome -----------------
    mordred_df = mordred_df.add_prefix("mrd_")
    feature_df = pd.concat([feature_df, mordred_df], axis=1)

    print(f"Mordred: {n_mordred_raw} descrittori calcolati")
    print(f"  {n_all_nan:4d} non calcolabili su nessuna molecola (tutti NaN) -> scartati")
    print(f"  {len(same_name):4d} con lo stesso nome di un descrittore RDKit:")
    print(f"       {len(identici)} con valori identici -> scartati (tengo la versione RDKit)")
    print(f"       {len(conflitti)} con valori DIVERSI -> tenuti come mrd_*: stesso nome, altra "
          f"implementazione")
    for c in conflitti:
        r = dup_name_df.set_index("descrittore").loc[c]
        print(f"         {c}: max|diff| = {r['max_diff_assoluta']:.4g}")
    print(f"  {len(dup_value_df):4d} identici a un descrittore RDKit sotto un altro nome -> scartati")
    if len(dup_value_df):
        print("       " + ", ".join(f"{r.mordred}={r.rdkit}" for r in dup_value_df.head(8).itertuples()))
    print(f"  {mordred_df.shape[1]:4d} colonne mrd_* aggiunte al pool")
    print(f"\nMatrice delle feature: {feature_df.shape}")

    log.table("mordred_duplicati_nome", dup_name_df, index=False)
    log.table("mordred_duplicati_valore", dup_value_df, index=False)
    log.result("mordred", {
        "n_calcolati": int(n_mordred_raw),
        "n_tutti_nan": int(n_all_nan),
        "n_stesso_nome": len(same_name),
        "n_stesso_nome_valori_uguali": len(identici),
        "n_stesso_nome_valori_diversi": len(conflitti),
        "stesso_nome_valori_diversi": conflitti,       # gli unici doppi che restano, di proposito
        "n_duplicati_per_valore": int(len(dup_value_df)),
        "n_tenuti": int(mordred_df.shape[1]),
    })

## 4. Split e filtro delle colonne

I filtri che usano statistiche delle colonne (come varianza o mediana) vengono adattati solo sui dati di training. In questo modo il test non influenza la scelta delle feature. Il transformer gestisce anche i valori mancanti: una colonna viene esclusa se supera MAX_NAN; altrimenti i valori mancanti vengono sostituiti con la mediana calcolata sul training set.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureFilter(BaseEstimator, TransformerMixin):
    """Scarta le colonne inutili e imputa i NaN guardando SOLO le righe di train.

    Bit di fingerprint (fp_*, maccs_*): filtro di prevalenza. VarianceThreshold(0) non basta,
    un bit acceso in una sola molecola su 139 ha varianza ~0.007 e sopravvive, ma da un solo
    esempio non si generalizza. Tengo i bit accesi tra il 2% e il 98% delle molecole di train.

    Descrittori continui: via le colonne costanti e quelle con piu di `max_nan` NaN (le
    tutte-NaN sono il caso limite). Quello che resta ha pochi NaN sparsi e li riempio con la
    mediana del train: sono descrittori non definiti su qualche molecola, non dati mancanti a
    caso, ma buttare l'intera colonna per una manciata di molecole costa piu di quanto renda.
    """

    def __init__(self, min_prev=MIN_PREV, max_nan=MAX_NAN):
        self.min_prev = min_prev
        self.max_nan = max_nan

    def fit(self, X, y=None):
        bin_cols = [c for c in X.columns if c.startswith(("fp_", "maccs_"))]
        cont_cols = [c for c in X.columns if c not in bin_cols]

        prev = X[bin_cols].mean()  # frazione di molecole di train in cui il bit e acceso
        self.keep_bin_ = prev[(prev >= self.min_prev) & (prev <= 1 - self.min_prev)].index.tolist()

        cont = X[cont_cols]
        cont = cont.loc[:, cont.isna().mean() <= self.max_nan]   # troppi NaN
        self.keep_cont_ = cont.columns[cont.std(ddof=0) > 0].tolist()

        # uso le mediane del solo train set
        self.medians_ = X[self.keep_cont_].median()
        self.selected_ = self.keep_cont_ + self.keep_bin_
        return self

    def transform(self, X):
        out = X[self.selected_].copy()
        out[self.keep_cont_] = out[self.keep_cont_].fillna(self.medians_)
        out[self.keep_bin_] = out[self.keep_bin_].fillna(0)   # non dovrebbe servire: le righe senza molecola sono gia' fuori
        return out                                            

# Costruisce la matrice delle feature
if MD_ONLY:
    # Solo descrittori MD: feature_df non esiste
    X = df[md_feats].copy()
    print(f"Modalita' MD-ONLY attiva: usate solo {X.shape[1]} colonne MD")
else:
    X = feature_df.reindex(df.index).copy()

# Pulizia e controlli comuni
X = X.replace([np.inf, -np.inf], np.nan)
y = df[target]

assert not X.isna().all(axis=1).any(), "righe tutte-NaN in X"

print("X:", X.shape, " y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

feat_filter = FeatureFilter().fit(X_train)   # fit sul solo train
X_train = feat_filter.transform(X_train)
X_test = feat_filter.transform(X_test)

n_dropped_nan = sum(
    c not in feat_filter.keep_cont_
    for c in X.columns if not c.startswith(("fp_", "maccs_"))
)
train_index_signature = hashlib.sha256("\n".join(sorted(X_train.index)).encode()).hexdigest()
test_index_signature = hashlib.sha256("\n".join(sorted(X_test.index)).encode()).hexdigest()
print(f"Train {X_train.shape} | Test {X_test.shape}")
print(f"Descrittori continui scartati (costanti o > {MAX_NAN:.0%} NaN): {n_dropped_nan}")
print(f"Colonne: {X.shape[1]} -> {X_train.shape[1]} "
      f"({len(feat_filter.keep_cont_)} continue + {len(feat_filter.keep_bin_)} binarie)")
print(f"{target} train: {y_train.mean():.0f} +/- {y_train.std():.0f} K | "
      f"{target} test: {y_test.mean():.0f} +/- {y_test.std():.0f} K")

log.result("split", {
    "test_size": 0.2,
    "n_train": int(X_train.shape[0]),
    "n_test": int(X_test.shape[0]),
    "n_cols_raw": int(X.shape[1]),
    "n_cols_after_filter": int(X_train.shape[1]),
    "n_cols_continue": len(feat_filter.keep_cont_),
    "n_cols_binarie": len(feat_filter.keep_bin_),
    "train_index_signature": train_index_signature,
    "test_index_signature": test_index_signature,
})

## 5. Selezione delle feature

La selezione avviene in due passaggi, mutual information e permutation importance, sempre dentro select_features(). La stessa procedura viene usata sia per il modello finale sia in ogni fold della nested cross-validation. Le feature MD non entrano in questa selezione, ma vengono aggiunte direttamente al modello full.


In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold

def select_features(X_tr, y_tr, n_mi=N_MI, n_splits=5, seed=random_state):
    """Selezione a due stadi, fittata solo su (X_tr, y_tr). Ritorna le colonne scelte.

    1. Mutual information mediata su n_splits fold -> tengo le top n_mi. La media serve a
       stabilizzare la stima di MI.
    2. Permutation importance out-of-fold: il modello e fittato sul train del fold e le
       importanze sono misurate sul validation, cosi' non premio le feature che servono solo a
       memorizzare. Tengo quelle sopra l'importanza media.
    """
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    discrete_mask = X_tr.columns.str.startswith(("fp_", "maccs_"))

    mi = np.zeros(X_tr.shape[1])

    for tr_idx, _ in cv.split(X_tr):
        mi += mutual_info_regression(
            X_tr.iloc[tr_idx],
            y_tr.iloc[tr_idx],
            discrete_features=discrete_mask,
            n_neighbors=5,
            random_state=seed
        )
    mi_series = pd.Series(mi / n_splits, index=X_tr.columns).sort_values(ascending=False)
    X_mi = X_tr[mi_series.head(n_mi).index]

    perm = np.zeros(X_mi.shape[1])
    for fit_idx, val_idx in cv.split(X_mi):
        m = GradientBoostingRegressor(random_state=seed).fit(X_mi.iloc[fit_idx], y_tr.iloc[fit_idx])
        perm += permutation_importance(
            m, X_mi.iloc[val_idx], y_tr.iloc[val_idx],
            scoring="neg_mean_absolute_error", n_repeats=10, random_state=seed,
        ).importances_mean
    perm_series = pd.Series(perm / n_splits, index=X_mi.columns).sort_values(ascending=False)

    return perm_series[perm_series > perm_series.mean()].index.tolist()

In [ ]:
# Prepara i modelli da confrontare: base e, se richiesto, full
split_arms = {}

if MD_ONLY:
    # Modalita di esecuzione: quali dati usare per prevedere il target
    X_train_md = X_train[[c for c in X_train.columns if c in md_feats]]
    X_test_md = X_test[[c for c in X_test.columns if c in md_feats]]
    split_arms["md_only"] = (X_train_md, X_test_md)
    print(f"Modalita'  MD-only: usate tutte le {X_train_md.shape[1]} colonne MD")
else:
    selected_feats = select_features(X_train, y_train)
    print(f"Feature descrittori strutturali selezionati: {len(selected_feats)} / {X_train.shape[1]}")
    print(selected_feats)
    # Braccio base: solo strutturali selezionate
    X_train_base = X_train[selected_feats]
    X_test_base = X_test[selected_feats]
    split_arms["base"] = (X_train_base, X_test_base)
    
    # Braccio full (se USE_MD): base + MD
    if USE_MD:
        X_train_full = with_md(X_train_base)
        X_test_full = with_md(X_test_base)
        split_arms["full"] = (X_train_full, X_test_full)

# Controlla che il join non abbia lasciato valori mancanti
for arm, (X_tr_, X_te_) in split_arms.items():
    for split, X_ in [("train", X_tr_), ("test", X_te_)]:
        assert not X_.isna().any().any(), f"{arm}/{split}: NaN dopo il join"

print("\n" + "   ".join(f"{arm} train {X_tr_.shape} / test {X_te_.shape}"
                        for arm, (X_tr_, X_te_) in split_arms.items()))

if MD_ONLY:
    log.result("selected_features_split", {
        "selection_applied": False,
        "n_selected": int(X_train_md.shape[1]),
        "n_candidate": int(X_train.shape[1]),
        "features": X_train_md.columns.tolist(),
    })
else:
    log.result("selected_features_split", {
        "selection_applied": True,
        "n_selected": len(selected_feats),
        "n_candidate": int(X_train.shape[1]),
        "features": selected_feats,
    })

## 6. Modelli sul singolo split

Questa sezione usa un solo split train/test per controlli e grafici, ad esempio il confronto tra valori osservati e predetti. Le prestazioni da usare per confrontare configurazioni diverse sono invece quelle della nested cross-validation nella sezione successiva. Anche i valori SHAP usati nell'analisi finale vengono calcolati out-of-fold nella nested cross-validation.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
}
cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
log.config(param_grid=param_grid)   

results = {}
for arm, (X_tr, X_te) in split_arms.items():
    gs = GridSearchCV(
        GradientBoostingRegressor(random_state=random_state),
        param_grid, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1,
    )
    gs.fit(X_tr, y_train)
    pred = gs.best_estimator_.predict(X_te)

    results[arm] = {
        "model": gs.best_estimator_,
        "pred": pred,
        "best_params": gs.best_params_,
        "cv_mae": -gs.best_score_,
        "test_mae": mean_absolute_error(y_test, pred),
        "test_r2": r2_score(y_test, pred),
    }
    r = results[arm]
    print(f"-- {ARM_LABEL[arm]} --")
    print(f"   {gs.best_params_}")
    print(f"   CV MAE {r['cv_mae]:.2f} | Test MAE {r['test_mae]:.2f} | Test R2 {r['test_r2']:.3f}")


# Salva i modelli con i nomi corrispondenti alla modalita scelta
if "base" in results:
    best_model_base, pred_base = results["base"]["model"], results["base"]["pred"]
if "md_only" in results:
    best_model_md_only, pred_md_only = results["md_only"]["model"], results["md_only"]["pred"]
if "full" in results:
    best_model_full, pred_full = results["full"]["model"], results["full"]["pred"]

log.result("single_split", {
    f"{arm}_{k}": results[arm][k]
    for arm in ARMS
    for k in ("best_params", "cv_mae", "test_mae", "test_r2")
})

In [ ]:
fig, axes = plt.subplots(1, len(ARMS), figsize=(6 * len(ARMS), 5), squeeze=False)
for ax, arm in zip(axes[0], ARMS):
    r = results[arm]
    pred_i = r["pred"]
    lims = [min(y_test.min(), pred_i.min()), max(y_test.max(), pred_i.max())]
    ax.scatter(y_test, pred_i, alpha=0.8, edgecolor="k", linewidth=0.3)
    ax.plot(lims, lims, "--", color="gray")
    ax.set_xlabel(f"{target} vera [K]")
    ax.set_ylabel(f"{target} predetta [K]")
    ax.set_title(f"{ARM_LABEL[arm]}  MAE={r['test_mae]:.1f}  R2={r['test_r2']:.3f}")
plt.tight_layout()
log.figure("single_split_pred_vs_true")
plt.show()

## 7. Nested cross-validation

Qui l'intera pipeline (filtro delle colonne, selezione delle feature e tuning) viene rieseguita dentro ogni fold esterno. Il modello viene poi valutato su dati che non ha mai visto. Quando sono disponibili le feature MD, i modelli base e full usano gli stessi fold: la differenza tra le loro prestazioni misura meglio il contributo delle MD, senza confonderlo con la variabilità dello split. Sugli stessi fold vengono calcolati anche i valori SHAP out-of-fold.


In [ ]:
from scipy import stats as st
import shap

outer_cv = KFold(n_splits=5, shuffle=True, random_state=random_state)
outer_splits = list(outer_cv.split(X))

nested_rows = {arm: [] for arm in ARMS}
fold_selected_feats = []                                # una lista di feature per fold -> stabilita'

oof_shap = {arm: [] for arm in ARMS}
oof_shap_X = {arm: [] for arm in ARMS}

for fold_i, (tr_idx, te_idx) in enumerate(outer_splits, start=1):
    X_tr_outer, X_te_outer = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr_outer, y_te_outer = y.iloc[tr_idx], y.iloc[te_idx]

    fold_filter = FeatureFilter().fit(X_tr_outer)       # filtro e selezione: solo train del fold
    X_tr_outer = fold_filter.transform(X_tr_outer)
    X_te_outer = fold_filter.transform(X_te_outer)

    sel = select_features(X_tr_outer, y_tr_outer)
    fold_selected_feats.append(list(sel))

    arms = {}
    
    if MD_ONLY:
        md_available = [c for c in md_feats if c in X_tr_outer.columns]
        arms["md_only"] = (X_tr_outer[md_available], X_te_outer[md_available])
    else:
        arms["base"] = (X_tr_outer[sel], X_te_outer[sel])
        
        # Braccio full se USE_MD = True
        if USE_MD:
            X_tr_full = pd.concat([
                X_tr_outer[sel],  # descrittori strutturali selezionati 
                df.loc[X_tr_outer.index, md_feats]  
            ], axis=1)
            X_te_full = pd.concat([
                X_te_outer[sel],  
                df.loc[X_te_outer.index, md_feats]  
            ], axis=1)
            
            arms["full"] = (X_tr_full, X_te_full)

    for arm, (X_tr_arm, X_te_arm) in arms.items():
        grid = GridSearchCV(
            GradientBoostingRegressor(random_state=random_state),
            param_grid, scoring="neg_mean_absolute_error",
            cv=KFold(n_splits=5, shuffle=True, random_state=random_state), n_jobs=-1,
        )
        grid.fit(X_tr_arm, y_tr_outer)
        # Modello del fold esterno, usato sia per la predizione OOF sia per SHAP sul relativo test fold
        fold_model = grid.best_estimator_
        pred = fold_model.predict(X_te_arm)

        # SHAP
        explainer = shap.TreeExplainer(fold_model)
        fold_shap = explainer.shap_values(X_te_arm)
        oof_shap[arm].append(
            pd.DataFrame(fold_shap, index=X_te_arm.index, columns=X_te_arm.columns)
        )
        oof_shap_X[arm].append(X_te_arm.copy())

        nested_rows[arm].append({
            "fold": fold_i,
            "n_features": X_tr_arm.shape[1],
            "best_params": grid.best_params_,
            "MAE": mean_absolute_error(y_te_outer, pred),
            "R2": r2_score(y_te_outer, pred),
        })

    print(f"Fold {fold_i}: " + " | ".join(
        f"{arm} n_feat={(r := nested_rows[arm][-1])['n_features']:3d}"
        f" MAE={r['MAE']:.2f} R2={r['R2']:.3f}" for arm in ARMS))

nested = {arm: pd.DataFrame(rows) for arm, rows in nested_rows.items()}

#  Estrai il dataframe base (o md_only se in modalita  MD_ONLY)
base_arm = "md_only" if MD_ONLY else "base"
nested_df = nested[base_arm]

print("\nNested CV (media +/- std tra i fold)")
for arm in ARMS:
    d = nested[arm]
    print(f"  {ARM_LABEL[arm]:20s} MAE {d['MAE'].mean():.2f} +/- {d['MAE'].std():.2f}"
          f"   R2 {d['R2'].mean():.3f} +/- {d['R2'].std():.3f}")

if USE_MD and not MD_ONLY:
    # contributo delle MD: confronto appaiato sugli stessi fold
    nested_df_full = nested["full"]
    d_mae = nested_df_full["MAE"].values - nested_df["MAE"].values
    t_stat, p_val = st.ttest_rel(nested_df_full["MAE"], nested_df["MAE"])
    print(f"\nDelta MAE (full - base): {d_mae.mean():+.2f} +/- {d_mae.std(ddof=1):.2f} (negativo = le MD aiutano)")
    print(f"paired t-test: t={t_stat:.2f}, p={p_val:.3f}")
    print("Con 5 fold la potenza e bassa: il p-value e indicativo, non una prova.")

# La stima da usare nei confronti finisce in runs/index.csv
nested_all = pd.concat([nested[arm].assign(arm=arm) for arm in ARMS])
log.table("nested_cv_folds", nested_all, index=False)

nested_summary = {"n_folds": outer_cv.get_n_splits()}
for arm in ARMS:
    d = nested[arm]
    nested_summary.update({
        f"{arm}_MAE_mean": float(d["MAE"].mean()),
        f"{arm}_MAE_std": float(d["MAE"].std()),
        f"{arm}_R2_mean": float(d["R2"].mean()),
        f"{arm}_R2_std": float(d["R2"].std()),
    })
log.result("nested_cv", nested_summary)

nested_all

In [ ]:
# Stabilita della selezione: quante volte ogni feature ricompare nei diversi fold
import itertools
from collections import Counter

counts = Counter(f for feats in fold_selected_feats for f in feats)
n_folds_sel = len(fold_selected_feats)

sel_freq = pd.DataFrame({
    "frequenza": pd.Series(counts, dtype=float) / n_folds_sel,
    "n_fold": pd.Series(counts, dtype=int),
}).sort_values("frequenza", ascending=False)
sel_freq["tipo"] = np.where(sel_freq.index.str.startswith(("fp_", "maccs_")), "binaria", "continua")

CORE_THR = 0.80   # core di consenso: feature scelte in almeno l'80% dei fold
consensus_core = sel_freq.index[sel_freq["frequenza"] >= CORE_THR].tolist()

# Similarita media tra le selezioni dei fold (Jaccard: 1.0 significa identiche)
sets = [set(f) for f in fold_selected_feats]
jac = np.mean([len(a & b) / len(a | b) for a, b in itertools.combinations(sets, 2)])

n_bin_core = sum(f.startswith(("fp_", "maccs_")) for f in consensus_core)
print(f"Feature scelte almeno una volta: {len(sel_freq)} su {n_folds_sel} fold")
print(f"Core di consenso (>= {CORE_THR:.0%} dei fold): {len(consensus_core)} feature "
      f"({len(consensus_core) - n_bin_core} continue + {n_bin_core} binarie)")
print(f"Jaccard medio tra fold: {jac:.3f} -> circa il {100 * (1 - jac):.0f}% della selezione "
      f"cambia da fold a fold.\n")
print(sel_freq.loc[consensus_core].to_string())

log.table("selection_stability", sel_freq.round(3))

log.result("selection_stability", {
    "n_folds": n_folds_sel,
    "core_threshold": CORE_THR,
    "n_selected_almeno_una_volta": len(sel_freq),
    "n_core": len(consensus_core),
    "jaccard_mean": float(jac),
    "consensus_core": consensus_core,

    # Mostra anche quante volte ogni feature e stata selezionata
    "selection_frequency": [
        (
            f"{feature}: {int(row['n_fold'])}/{n_folds_sel} "
            f"({row['frequenza']:.2f}) [{row['tipo']}]"
        )
        for feature, row in sel_freq.iterrows()
    ],
})

top = sel_freq.head(30).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, max(4, 0.28 * len(top))))
ax.barh(top.index, top["frequenza"],
        color=["#c44e52" if t == "binaria" else "#4c72b0" for t in top["tipo"]])
ax.axvline(CORE_THR, ls="--", c="k", lw=1, label=f"soglia core ({CORE_THR:.0%})")
ax.set_xlabel(f"frazione dei {n_folds_sel} fold in cui la feature e selezionata")
ax.set_title("Stabilita' della selezione (blu = continua, rosso = bit)")
ax.legend()
plt.tight_layout()
log.figure("selection_stability")
plt.show()

In [ ]:
# ============================================================
# Core congiunto di descrittori strutturali e feature MD (solo analisi esplorativa)
# Questo core non viene usato per addestrare i modelli base o full.
# ============================================================

if not USE_MD or MD_ONLY:
    print("Core congiunto disponibile solo in MODE='struct+md'")

else:
    joint_fold_selected_feats = []

    for fold_i, (tr_idx, _) in enumerate(outer_splits, start=1):
        y_tr = y.iloc[tr_idx]

        # Pool completo: descrittori strutturali + tutte le MD
        X_joint_raw = pd.concat([
            X.iloc[tr_idx],
            df.loc[X.index[tr_idx], md_feats]
        ], axis=1)

        # Stesso filtro, fittato esclusivamente sul train del fold
        joint_filter = FeatureFilter().fit(X_joint_raw)
        X_joint = joint_filter.transform(X_joint_raw)

        # Stessa feature selection della pipeline
        joint_sel = select_features(X_joint, y_tr)
        joint_fold_selected_feats.append(list(joint_sel))

        n_md = sum(f in md_feats for f in joint_sel)
        print(
            f"Fold {fold_i}: {len(joint_sel)} selezionate "
            f"({len(joint_sel) - n_md} strutturali + {n_md} MD)"
        )

    # Frequenza di selezione
    joint_counts = Counter(
        f for feats in joint_fold_selected_feats for f in feats
    )

    joint_sel_freq = pd.DataFrame({
        "frequenza": pd.Series(joint_counts, dtype=float) / len(joint_fold_selected_feats),
        "n_fold": pd.Series(joint_counts, dtype=int),
    }).sort_values("frequenza", ascending=False)

    joint_sel_freq["tipo"] = np.where(
        joint_sel_freq.index.isin(md_feats),
        "MD",
        "strutturale"
    )

    # Core congiunto
    joint_consensus_core = joint_sel_freq.index[
        joint_sel_freq["frequenza"] >= CORE_THR
    ].tolist()

    joint_structural_core = [
        f for f in joint_consensus_core if f not in md_feats
    ]
    joint_md_core = [
        f for f in joint_consensus_core if f in md_feats
    ]

    # Stabilita  interna del nuovo core
    joint_sets = [set(x) for x in joint_fold_selected_feats]
    joint_jaccard = np.mean([
        len(a & b) / len(a | b)
        for a, b in itertools.combinations(joint_sets, 2)
    ])

    print(f"\nCore congiunto >= {CORE_THR:.0%}: {len(joint_consensus_core)}")
    print(f"  strutturali: {len(joint_structural_core)}")
    print(f"  MD:          {len(joint_md_core)}")
    print(f"Jaccard medio: {joint_jaccard:.3f}")

    print("\nMD nel core congiunto:")
    print(joint_sel_freq.loc[joint_md_core].to_string()
          if joint_md_core else "nessuna")

    print("\nFeature strutturali nel core congiunto:")
    print(joint_sel_freq.loc[joint_structural_core].to_string())
    original_core = set(consensus_core)

    joint_struct_core = set(joint_structural_core)

    stable_in_both = sorted(original_core & joint_struct_core)
    lost_with_md = sorted(original_core - joint_struct_core)
    new_structural = sorted(joint_struct_core - original_core)

    print("\n=== Confronto core strutturale vs core congiunto ===")

    print("\nRestano stabili in entrambi:")
    print(stable_in_both)

    print("\nNel core strutturale ma non nel core congiunto:")
    print(lost_with_md)

    print("\nEntrano nel core strutturale solo quando competono anche le MD:")
    print(new_structural)

    print("\nMD nel core congiunto:")
    print(joint_md_core)

    core_comparison = pd.concat([
        sel_freq["frequenza"].rename("freq_core_strutturale"),
        joint_sel_freq["frequenza"].rename("freq_core_congiunto"),
    ], axis=1)

    core_comparison["tipo"] = np.where(
        core_comparison.index.isin(md_feats),
        "MD",
        "strutturale"
    )

    core_comparison["nel_core_strutturale"] = (
        core_comparison.index.isin(consensus_core)
    )

    core_comparison["nel_core_congiunto"] = (
        core_comparison.index.isin(joint_consensus_core)
    )

    core_comparison = core_comparison.sort_values(
        "freq_core_congiunto",
        ascending=False
    )

    core_comparison

    # ============================================================
    # Salvataggio dei risultati del core congiunto nel report
    # ============================================================

    log.table(
        "joint_selection_stability",
        joint_sel_freq.round(3)
    )

    log.table(
        "core_structural_vs_joint",
        core_comparison.round(3)
    )

    log.result("joint_selection_stability", {
        "analysis_type": "exploratory_only",
        "used_for_training_full_arm": False,
        "n_folds": len(joint_fold_selected_feats),
        "core_threshold": CORE_THR,
        "jaccard_mean": float(joint_jaccard),

        "n_core": len(joint_consensus_core),
        "n_structural_core": len(joint_structural_core),
        "n_md_core": len(joint_md_core),

        "consensus_core_joint": joint_consensus_core,
        "structural_core_joint": joint_structural_core,
        "md_core_joint": joint_md_core,

        "stable_in_both": stable_in_both,
        "lost_with_md": lost_with_md,
        "new_structural_with_md": new_structural,

        "selection_frequency": [
            (
                f"{feature}: {int(row['n_fold'])}/{len(joint_fold_selected_feats)} "
                f"({row['frequenza']:.2f}) [{row['tipo']}]"
            )
            for feature, row in joint_sel_freq.iterrows()
        ],
    })

In [ ]:
# ============================================================
# Ablation: training con selezione congiunta strutturali + MD
# ============================================================

if not USE_MD or MD_ONLY:
    print("Ablation disponibile solo in MODE='struct+md'")

else:
    joint_train_rows = []
    joint_train_selected_feats = []

    for fold_i, (tr_idx, te_idx) in enumerate(outer_splits, start=1):
        y_tr_outer, y_te_outer = y.iloc[tr_idx], y.iloc[te_idx]

        # Pool congiunto: strutturali + MD
        X_tr_raw = pd.concat([
            X.iloc[tr_idx],
            df.loc[X.index[tr_idx], md_feats]
        ], axis=1)

        X_te_raw = pd.concat([
            X.iloc[te_idx],
            df.loc[X.index[te_idx], md_feats]
        ], axis=1)

        # Filtro fittato solo sul training fold
        joint_filter = FeatureFilter().fit(X_tr_raw)
        X_tr_joint = joint_filter.transform(X_tr_raw)
        X_te_joint = joint_filter.transform(X_te_raw)

        # Qui la selezione congiunta viene usata davvero per il training
        joint_sel = select_features(X_tr_joint, y_tr_outer)
        joint_train_selected_feats.append(list(joint_sel))

        X_tr_joint = X_tr_joint[joint_sel]
        X_te_joint = X_te_joint[joint_sel]

        # Stesso tuning della nested CV principale
        grid = GridSearchCV(
            GradientBoostingRegressor(random_state=random_state),
            param_grid,
            scoring="neg_mean_absolute_error",
            cv=KFold(
                n_splits=5,
                shuffle=True,
                random_state=random_state
            ),
            n_jobs=-1,
        )

        grid.fit(X_tr_joint, y_tr_outer)

        fold_model = grid.best_estimator_
        pred = fold_model.predict(X_te_joint)

        selected_md = [f for f in joint_sel if f in md_feats]

        joint_train_rows.append({
            "fold": fold_i,
            "n_features": len(joint_sel),
            "n_structural": len(joint_sel) - len(selected_md),
            "n_md": len(selected_md),
            "best_params": grid.best_params_,
            "MAE": mean_absolute_error(y_te_outer, pred),
            "R2": r2_score(y_te_outer, pred),
            "selected_features": ", ".join(joint_sel),
            "md_features": ", ".join(selected_md),
        })

        r = joint_train_rows[-1]
        print(
            f"Fold {fold_i}: "
            f"n_feat={r['n_features']:3d} "
            f"({r['n_structural']} strutturali + {r['n_md']} MD) "
            f"MAE={r['MAE']:.2f} R2={r['R2']:.3f}"
        )

    joint_training_df = pd.DataFrame(joint_train_rows)

    # ========================================================
    # Confronto con base e full sugli stessi outer fold
    # ========================================================

    base_mae = nested["base"]["MAE"].values
    full_mae = nested["full"]["MAE"].values
    joint_mae = joint_training_df["MAE"].values

    d_joint_base = joint_mae - base_mae
    d_joint_full = joint_mae - full_mae

    t_base, p_base = st.ttest_rel(joint_mae, base_mae)
    t_full, p_full = st.ttest_rel(joint_mae, full_mae)

    print("\nTraining con selezione congiunta")
    print(
        f"  MAE {joint_training_df['MAE'].mean():.2f} "
        f"+/- {joint_training_df['MAE'].std():.2f}"
    )
    print(
        f"  R2  {joint_training_df['R2'].mean():.3f} "
        f"+/- {joint_training_df['R2'].std():.3f}"
    )

    print(
        f"\nDelta MAE joint - base: "
        f"{d_joint_base.mean():+.2f} +/- {d_joint_base.std(ddof=1):.2f}"
    )
    print(
        f"Delta MAE joint - full: "
        f"{d_joint_full.mean():+.2f} +/- {d_joint_full.std(ddof=1):.2f}"
    )

    print(f"paired t-test vs base: t={t_base:.2f}, p={p_base:.3f}")
    print(f"paired t-test vs full: t={t_full:.2f}, p={p_full:.3f}")
    print("Con 5 fold i p-value sono solo indicativi.")

    # ========================================================
    # Logger
    # ========================================================

    log.table(
        "joint_selection_training_folds",
        joint_training_df,
        index=False
    )

    log.result("joint_selection_training", {
        "analysis_type": "ablation_training",
        "joint_selection_used_for_training": True,
        "n_folds": len(joint_training_df),

        "MAE_mean": float(joint_training_df["MAE"].mean()),
        "MAE_std": float(joint_training_df["MAE"].std()),
        "R2_mean": float(joint_training_df["R2"].mean()),
        "R2_std": float(joint_training_df["R2"].std()),

        "selected_features_per_fold": [
            f"fold {i}: {', '.join(features)}"
            for i, features in enumerate(
                joint_train_selected_feats,
                start=1
            )
        ],

        "md_features_per_fold": [
            (
                f"fold {i}: "
                f"{', '.join(f for f in features if f in md_feats) or '(nessuna)'}"
            )
            for i, features in enumerate(
                joint_train_selected_feats,
                start=1
            )
        ],
    })

    joint_training_df

## 8. Quanto aiutano le feature MDe

Questa sezione analizza quali informazioni usa il modello: il peso complessivo delle feature MD, l'importanza delle singole colonne e la stabilità della selezione tra i fold. Le spiegazioni SHAP arrivano dalla nested cross-validation. Se USE_MD = False, le analisi specifiche sulle feature MD vengono saltate.


In [ ]:
import shap

fig, axes = plt.subplots(
    1, len(ARMS),
    figsize=(7 * len(ARMS), 5),
    squeeze=False
)

shap_by_arm = {}
X_shap_by_arm = {}

for ax, arm in zip(axes[0], ARMS):

    # Valori SHAP sui test fold
    shap_df = pd.concat(oof_shap[arm], axis=0)

    # Valori delle feature sugli stessi campioni
    X_df = pd.concat(oof_shap_X[arm], axis=0)

    # Insieme delle feature viste nei fold
    all_features = shap_df.columns.union(X_df.columns)

    shap_df = shap_df.reindex(columns=all_features).fillna(0) # I descrittori non usati nei fold risultano NaN, li porto a 0
    X_df = X_df.reindex(columns=all_features)

    shap_by_arm[arm] = shap_df
    X_shap_by_arm[arm] = X_df

    plt.sca(ax)
    shap.summary_plot(shap_df.values, X_df, show=False)

    ax.set_title(f"SHAP summary - {ARM_LABEL[arm]}")

plt.tight_layout()
log.figure("shap_summary_per_arm")
plt.show()

base_arm = "md_only" if MD_ONLY else "base"
shap_base = shap_by_arm[base_arm]

if USE_MD and not MD_ONLY:
    shap_full = shap_by_arm["full"]

shap_feature_importance = {}

for arm in ARMS:

    shap_df = shap_by_arm[arm]
    rows = []

    for feature in shap_df.columns:

        # Importanza globale OOF, inclusi gli 0 dei fold
        # in cui la feature strutturale non era presente
        mean_oof = shap_df[feature].abs().mean()

        # SHAP della feature solo nei fold in cui esisteva
        values_when_present = [
            fold[feature].abs()
            for fold in oof_shap[arm]
            if feature in fold.columns
        ]

        n_folds_present = len(values_when_present)

        if values_when_present:
            mean_when_present = pd.concat(values_when_present).mean()
        else:
            mean_when_present = np.nan

        rows.append({
            "feature": feature,
            "folds_present": n_folds_present,
            "mean_abs_shap_oof": mean_oof,
            "mean_abs_shap_when_present": mean_when_present,
        })

    importance = (
        pd.DataFrame(rows)
        .set_index("feature")
        .sort_values("mean_abs_shap_oof", ascending=False)
    )

    shap_feature_importance[arm] = importance

    log.table(
        f"shap_oof_feature_importance_{arm}",
        importance.round(4)
    )

### 8.1 Importanza congiunta delle MD

In [ ]:
if not USE_MD or MD_ONLY:
    print("Analisi disponibile solo nel confronto RDKit vs RDKit+MD")
else:
    # Colonne direttamente dallo SHAP OOF del braccio full
    md_cols = [c for c in md_feats if c in shap_full.columns]
    struct_cols = [c for c in shap_full.columns if c not in md_cols]

    # 1. SHAP di gruppo:
    # per ogni campione somma |SHAP| delle feature del gruppo,
    # poi fa la media sui campioni OOF
    shap_md_joint = shap_full[md_cols].abs().sum(axis=1).mean()
    shap_struct_joint = shap_full[struct_cols].abs().sum(axis=1).mean()

    shap_total = shap_md_joint + shap_struct_joint

    print("SHAP di gruppo")
    print(f"  MD          : {shap_md_joint:.3f}  "
          f"({100 * shap_md_joint / shap_total:.1f}%)")
    print(f"  Strutturali : {shap_struct_joint:.3f}  "
          f"({100 * shap_struct_joint / shap_total:.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    bars = axes[0].bar(
        ["Strutturali", "MD"],
        [shap_struct_joint, shap_md_joint],
        color=["steelblue", "darkorange"]
    )

    axes[0].bar_label(
        bars,
        labels=[
            f"{v:.2f}\n({100 * v / shap_total:.1f}%)"
            for v in (shap_struct_joint, shap_md_joint)
        ],
        padding=4
    )

    axes[0].set_ylabel("mean(|SHAP|) sommato sul gruppo")
    axes[0].set_title("Quota dell'importanza SHAP assoluta per gruppo")

    # 2. Importanza delle singole feature MD
    md_shap = (
        shap_full[md_cols]
        .abs()
        .mean(axis=0)
        .sort_values()
    )

    md_shap.plot(
        kind="barh",
        ax=axes[1],
        color="darkorange"
    )

    axes[1].set_xlabel("mean(|SHAP|)")
    axes[1].set_title("Importanza delle singole MD")

    plt.tight_layout()
    log.figure("md_importance")
    plt.show()

    log.table(
        "md_shap_importance",
        md_shap.rename("mean_abs_shap").to_frame().round(4)
    )

    log.result("md_importance", {
        "md_cols": md_cols,
        "shap_joint_md": float(shap_md_joint),
        "shap_joint_struct": float(shap_struct_joint),
        "shap_share_md_pct": float(
            100 * shap_md_joint / shap_total
        )
    })

## 9. Modello finale

La pipeline completa viene eseguita una sola volta su tutti i dati per ottenere una lista finale di feature e un modello da riutilizzare. Questo modello non serve a stimare le prestazioni: ha visto tutte le righe e quindi non c'è un test set indipendente. Per il confronto tra configurazioni si usano le metriche della nested cross-validation. 


In [ ]:
final_filter = FeatureFilter().fit(X)
X_filt = final_filter.transform(X)

if MD_ONLY:
    # In MD-only le feature MD sono poche: niente MI o permutation selection.
    final_feats = [c for c in md_feats if c in X_filt.columns]
else:
    final_feats = select_features(X_filt, y)

X_final = X_filt[final_feats]

final_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=random_state),
    param_grid, scoring="neg_mean_absolute_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=random_state), n_jobs=-1,
)
final_grid.fit(X_final, y)
final_model = final_grid.best_estimator_

n_bin = sum(f.startswith(("fp_", "maccs_")) for f in final_feats)
print(f"Colonne: {X.shape[1]} -> {X_filt.shape[1]} (filtro) -> {len(final_feats)} (selezione)")
print(f"Feature finali: {len(final_feats) - n_bin} continue + {n_bin} binarie")
print(f"Iperparametri: {final_grid.best_params_}")

# Controllo di coerenza: quanto la lista finale coincide con il core stabile
in_core = [f for f in final_feats if f in consensus_core]
print(f"\nCore di consenso presente nel modello finale: {len(in_core)}/{len(consensus_core)}")
print(f"Feature finali fuori dal core (scelte da questo fit, meno stabili): {len(final_feats) - len(in_core)}")
print(final_feats)

log.result("final_model", {
    "n_cols_raw": int(X.shape[1]),
    "n_cols_after_filter": int(X_filt.shape[1]),
    "n_features": len(final_feats),
    "n_features_continue": len(final_feats) - n_bin,
    "n_features_binarie": n_bin,
    "best_params": final_grid.best_params_,
    "features": final_feats,
    "core_presente": f"{len(in_core)}/{len(consensus_core)}",
})

## 10. UMAP e clustering

UMAP serve a visualizzare la struttura dello spazio delle feature e a controllare se emergono cluster. Il modello usato qui è quello addestrato sull'intero dataset.


In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Feature usate da UMAP:
# - senza MD -> sole feature strutturali
# - con MD   -> feature strutturali + tutte le feature MD
# - MD only  -> sole feature MD

X_struct = X_filt.copy()

if USE_MD:
    X_md = df.loc[X_struct.index, md_feats].copy()
    X_md = X_md.replace([np.inf, -np.inf], np.nan)
    X_md = X_md.fillna(X_md.median())

    if MD_ONLY:
        X_proj = X_md
        umap_space = "md_only"
    else:
        X_proj = pd.concat([X_struct, X_md], axis=1)
        umap_space = "structural+md"
else:
    X_proj = X_struct
    umap_space = "structural"

# per eventuali NaN residui
X_proj = X_proj.fillna(X_proj.median())

X_scaled = pd.DataFrame(
    StandardScaler().fit_transform(X_proj),
    columns=X_proj.columns,
    index=X_proj.index
)

print(
    f"UMAP space: {umap_space} | "
    f"{X_proj.shape[0]} polimeri | "
    f"{X_proj.shape[1]} feature"
)

if USE_MD:
    print(f"Feature MD incluse ({len(md_feats)}): {md_feats}")

X_scaled = pd.DataFrame(StandardScaler().fit_transform(X_proj), columns=X_proj.columns, index=X_proj.index)

X_umap = umap.UMAP(n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST, random_state=random_state).fit_transform(X_scaled)

scores = {}

for k in UMAP_K_VALUES:
    labels_k = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=random_state
    ).fit_predict(X_umap)

    scores[k] = silhouette_score(X_umap, labels_k)

best_k = max(scores, key=scores.get)
best_silhouette = scores[best_k]

print("Silhouette score:")
for k, score in scores.items():
    print(f"k={k}: {score:.3f}")

print(
    f"\nNumero di cluster scelto: k={best_k} "
    f"(silhouette={best_silhouette:.3f})"
)

# Raggruppa i campioni nello spazio UMAP
labels = KMeans(n_clusters=best_k, n_init=10, random_state=random_state).fit_predict(X_umap)

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=y, alpha=0.8, edgecolor="k", linewidth=0.3)
plt.colorbar(sc, label=f"{target} [K]")
ax.set_title(f"UMAP, colore = {target}")
plt.tight_layout()
log.figure("umap")
plt.show()

df_umap = pd.DataFrame(X_umap, columns=["UMAP1", "UMAP2"], index=X.index)
df_umap["cluster"] = labels
df_umap[target] = y
print(df_umap.groupby("cluster")[target].describe().round(1))

# Confronta i cluster tramite la differenza tra le medie standardizzate
cluster_means = X_scaled.assign(cluster=labels).groupby("cluster").mean()
cluster_sep = (cluster_means.max(axis=0) -cluster_means.min(axis=0)).sort_values(ascending=False)
print("\nFeature che piu separano i cluster:")
print(cluster_sep.head(10).round(2))

log.table("umap_clusters", df_umap.groupby("cluster")[target].describe().round(1))

log.result("umap", {
    "n_neighbors": UMAP_N_NEIGHBORS,
    "min_dist": UMAP_MIN_DIST,
    "clustering_space": "UMAP",
    "cluster_method": "KMeans",
    "k_selection": "silhouette",

    "k_testati": UMAP_K_VALUES,
    "silhouette_scores": {
        int(k): round(float(v), 4)
        for k, v in scores.items()
    },

    "n_clusters": int(best_k),
    "best_silhouette": round(float(best_silhouette), 4),

    "cluster_sizes": (df_umap["cluster"].value_counts().sort_index().to_dict()),

    f"{target}_medio_per_cluster": (
        df_umap.groupby("cluster")[target]
        .mean()
        .round(1)
        .to_dict()
    ),

    "top_feature_separanti": (cluster_sep.head(10).round(2).to_dict()),
})

In [ ]:
# Grafico UMAP con i cluster evidenziati
from scipy.spatial import ConvexHull
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(6.5, 5))

# Stessi punti del grafico UMAP originale
sc = ax.scatter(
    X_umap[:, 0], X_umap[:, 1],
    c=y, alpha=0.8,
    edgecolor="k", linewidth=0.3
)
plt.colorbar(sc, label=f"{target} [K]")

# Disegna solo il contorno dei cluster
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

for cluster in np.unique(labels):
    points = X_umap[labels == cluster]

    hull = ConvexHull(points)
    boundary = points[hull.vertices]
    boundary = np.vstack([boundary, boundary[0]])  # chiude il contorno

    ax.plot(
        boundary[:, 0],
        boundary[:, 1],
        linewidth=2,
        color=colors[cluster],
        label=f"Cluster {cluster}"
    )

ax.set_title(f"UMAP con contorno dei cluster")
ax.legend()
plt.tight_layout()

log.figure("umap_clusters_outline")
plt.show()

## 11. Chiusura del run

log.finish() salva il report leggibile del run e aggiunge una riga a runs/index.csv. Il run.json viene aggiornato durante l'esecuzione, così anche un notebook interrotto conserva i risultati già prodotti.


In [ ]:
# Nota libera sul run: utile per ricordare perche e stata scelta questa configurazione.

if USE_MD:
    log.note(f"MD ({len(md_feats)}): {', '.join(md_feats)} da {md_source}")
else:
    log.note("Run senza feature MD: sola pipeline RDKit.")

if USE_MORDRED:
    log.note(f"Descrittori Mordred attivi: {sum(c.startswith('mrd_') for c in X.columns)} colonne "
             f"mrd_* nel pool, al netto dei duplicati con RDKit.")

log.finish()

In [ ]:
# Elenco dei run eseguiti

runs = load_runs()
cols = ["run_id", "collection", "n_polymers", "use_md", "n_md_feats", "use_mordred",
        "n_mordred_feats", "n_final_feats", "nested_MAE_base", "nested_MAE_full",
        "delta_MAE_full_base", "delta_MAE_pvalue"]
runs[[c for c in cols if c in runs.columns]]